In [455]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, accuracy_score
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from xgboost import XGBRegressor, XGBClassifier

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam

In [456]:
cutoff_date = pd.Timestamp('2021-10-19', tz= 'UTC')

In [457]:
df_1h= pd.read_parquet("1h_btc_sentiment.parquet")

In [458]:
df_15min= pd.read_parquet("15min_btc_sentiment.parquet")

In [459]:
df_cluster_1h= pd.read_parquet("1h_btc_sentiment_cluster.parquet")

In [495]:
test= pd.read_parquet("sentiment_hourly_aggregated.parquet")

In [496]:
test.head(10)

,avg_score,n_positive,n_negative,total_tweets,sentiment_mode,sentiment_median,net_sentiment
datetime,,,,,,,
2021-01-01 00:00:00,0.469112,18,876,1272,negative,None,-858
2021-01-01 01:00:00,0.472049,7,946,1373,negative,None,-939
2021-01-01 02:00:00,0.473136,6,747,1102,negative,None,-741
2021-01-01 03:00:00,0.471808,4,629,972,negative,None,-625
2021-01-01 04:00:00,0.468677,3,552,840,negative,None,-549
2021-01-01 05:00:00,0.467536,7,509,766,negative,None,-502
2021-01-01 06:00:00,0.470887,7,540,790,negative,None,-533
2021-01-01 07:00:00,0.467385,3,432,677,negative,None,-429
2021-01-01 08:00:00,0.466856,0,493,786,negative,None,-493


In [460]:
df_cluster_15min= pd.read_parquet("15min_btc_sentiment_cluster.parquet")

In [461]:
keep_base = ["return", "return_class"]

keep_cluster = [
    c for c in df_cluster_1h.columns
    if c.startswith("sentiment_mean_c") or c.startswith("sentiment_weighted_c")
]

keep_sentiment= ["sentiment_mean", "sentiment_weighted"]

In [462]:
#df= df_1h[keep_base + keep_sentiment].copy()
#df= df_15min[keep_base + keep_sentiment].copy()
df= df_cluster_1h[keep_base + keep_cluster].copy()
#df= df_cluster_15min[keep_base + keep_cluster].copy()

In [463]:
df.columns

Index(['return', 'return_class', 'sentiment_mean_c0.0', 'sentiment_mean_c1.0',
       'sentiment_mean_c2.0', 'sentiment_mean_c3.0', 'sentiment_mean_c4.0',
       'sentiment_mean_c5.0', 'sentiment_mean_c6.0', 'sentiment_mean_c7.0',
       'sentiment_mean_c8.0', 'sentiment_mean_c9.0', 'sentiment_mean_c10.0',
       'sentiment_mean_c11.0', 'sentiment_mean_c12.0', 'sentiment_mean_c13.0',
       'sentiment_mean_c14.0', 'sentiment_mean_c15.0', 'sentiment_mean_c16.0',
       'sentiment_mean_c17.0', 'sentiment_mean_c18.0', 'sentiment_mean_c19.0',
       'sentiment_mean_c20.0', 'sentiment_mean_c21.0', 'sentiment_mean_c22.0',
       'sentiment_mean_c23.0', 'sentiment_mean_c24.0',
       'sentiment_weighted_c0.0', 'sentiment_weighted_c1.0',
       'sentiment_weighted_c2.0', 'sentiment_weighted_c3.0',
       'sentiment_weighted_c4.0', 'sentiment_weighted_c5.0',
       'sentiment_weighted_c6.0', 'sentiment_weighted_c7.0',
       'sentiment_weighted_c8.0', 'sentiment_weighted_c9.0',
       'sentimen

In [464]:
df['volatility_4h'] = df['return'].rolling('4h').std()
df['volatility_12h'] = df['return'].rolling('12h').std()
df['volatility_24h'] = df['return'].rolling('24h').std()
df['mean_return_4h'] = df['return'].rolling('4h').mean()
df['mean_return_12h'] = df['return'].rolling('12h').mean()
df['mean_return_24h'] = df['return'].rolling('24h').mean()
df['hour_sin'] = np.sin(2*np.pi*df.index.hour / 24)
df['hour_cos'] = np.cos(2*np.pi*df.index.hour / 24)
df['dow_sin'] = np.sin(2*np.pi*df.index.dayofweek / 7)
df['dow_cos'] = np.cos(2*np.pi*df.index.dayofweek / 7)
df['target_return'] = df['return'].shift(-1)
df['target_return_class'] = df['return_class'].shift(-1)

In [465]:
base_features = [
    'return',
    'volatility_4h',
    'volatility_12h',
    'volatility_24h',
    'mean_return_4h',
    'mean_return_12h',
    'mean_return_24h',
    'hour_sin',
    'hour_cos',
    'dow_sin',
    'dow_cos',
]


sent_mean_cols = [c for c in df.columns if c.startswith("sentiment_mean_c")]
sent_weight_cols = [c for c in df.columns if c.startswith("sentiment_weighted_c")]

#features = base_features + ["sentiment_weighted"]#["sentiment_mean"]
features = base_features + sent_weight_cols#sent_mean_cols


In [466]:
needed= features+ ["target_return", "target_return_class"]
df = df.dropna(subset=needed).copy()

In [467]:
X= df[features]
y_reg= df['target_return']
y_cla= df['target_return_class']

In [468]:
X_train= X.loc[X.index <= cutoff_date]
X_test= X.loc[X.index > cutoff_date]

y_train_reg= y_reg.loc[y_reg.index <= cutoff_date]
y_test_reg= y_reg.loc[y_reg.index > cutoff_date]

y_train_cla= y_cla.loc[y_cla.index <= cutoff_date]
y_test_cla= y_cla.loc[y_cla.index > cutoff_date]

In [469]:
lin_model_reg = LinearRegression().fit(X_train, y_train_reg)


In [470]:
log_model_clf = LogisticRegression().fit(X_train, y_train_cla)

In [471]:
lin_pred_reg = lin_model_reg.predict(X_test)
log_pred_cla = log_model_clf.predict(X_test)

In [472]:
print(f"Linear MAE (Next return prediction): {mean_absolute_error(y_test_reg, lin_pred_reg):.6f}")
print(f"Logistic Accuracy (Classification prediction): {accuracy_score(y_test_cla, log_pred_cla):.4f}")

Linear MAE (Next return prediction): 0.004724
Logistic Accuracy (Classification prediction): 0.3331


In [473]:
rfr_model_reg = RandomForestRegressor(
    n_estimators=50,
    max_depth=6,
    n_jobs=-1,
    random_state=42
)
rfr_model_reg.fit(X_train, y_train_reg)

RandomForestRegressor(max_depth=6, n_estimators=50, n_jobs=-1, random_state=42)

In [474]:
rfc_model_cla = RandomForestClassifier(
    n_estimators=50,
    max_depth=6,
    n_jobs=-1,
    random_state=42
)
rfc_model_cla.fit(X_train, y_train_cla)

RandomForestClassifier(max_depth=6, n_estimators=50, n_jobs=-1, random_state=42)

In [475]:
rfr_pred_reg = rfr_model_reg.predict(X_test)
rfc_pred_cla = rfc_model_cla.predict(X_test)

In [476]:
print(f"Random Forest Regressor MAE (Next return prediction): {mean_absolute_error(y_test_reg, rfr_pred_reg):.6f}")
print(f"Random Forest Classifier Accuracy (Classification prediction): {accuracy_score(y_test_cla, rfc_pred_cla):.4f}")

Random Forest Regressor MAE (Next return prediction): 0.004702
Random Forest Classifier Accuracy (Classification prediction): 0.3663


In [ ]:
XGB_model_reg = XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    objective='reg:squarederror'
)
XGB_model_reg.fit(X_train, y_train_reg)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.05, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=6, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=200, n_jobs=None,
             num_parallel_tree=None, random_state=None, ...)

In [478]:
XGB_model_cla = XGBClassifier(
    n_estimators=150,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8
)
XGB_model_cla.fit(X_train, y_train_cla)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=150, n_jobs=None,
              num_parallel_tree=None, objective='multi:softprob', ...)

In [479]:
XGB_pred_reg = XGB_model_reg.predict(X_test)
XGB_pred_cla = XGB_model_cla.predict(X_test)

In [480]:
print(f"XG Boost Regressor MAE (Next return prediction): {mean_absolute_error(y_test_reg, rfr_pred_reg):.6f}")
print(f"XG Boost Classifier Accuracy (Classification prediction): {accuracy_score(y_test_cla, rfc_pred_cla):.4f}")

XG Boost Regressor MAE (Next return prediction): 0.004702
XG Boost Classifier Accuracy (Classification prediction): 0.3663


In [481]:
X_val = df[features].values
y_reg_val = df['target_return'].values
y_cla_val = df['target_return_class'].values

In [482]:
def create_sequences(X, y_r, y_c, window=20):
    Xs, yr, yc = [], [], []
    for i in range(window, len(X)):
        Xs.append(X[i-window:i])
        yr.append(y_r[i])
        yc.append(y_c[i])
    return np.array(Xs), np.array(yr), np.array(yc)

In [483]:
WINDOW= 20

In [484]:
X_seq, y_reg_seq, y_clf_seq = create_sequences(X_val, y_reg_val, y_cla_val, WINDOW)

In [485]:
seq_index = df.index[WINDOW:]

In [486]:
train_mask = seq_index < cutoff_date
test_mask  = seq_index >= cutoff_date

In [487]:
X_train_seq = X_seq[train_mask]
X_test_seq  = X_seq[test_mask]

y_train_reg_seq = y_reg_seq[train_mask]
y_test_reg_seq  = y_reg_seq[test_mask]

y_train_cla_seq = y_clf_seq[train_mask]
y_test_cla_seq  = y_clf_seq[test_mask]

In [488]:
scaler = MinMaxScaler()
n_feat = X_train_seq.shape[-1]

X_train_seq_2d = X_train_seq.reshape(-1, n_feat)
X_test_seq_2d  = X_test_seq.reshape(-1, n_feat)

X_train_seq_scaled = scaler.fit_transform(X_train_seq_2d).reshape(X_train_seq.shape)
X_test_seq_scaled  = scaler.transform(X_test_seq_2d).reshape(X_test_seq.shape)

In [489]:
gru_model_reg = Sequential([
    Input(shape=(WINDOW, len(features))),
    GRU(32),
    Dropout(0.2),
    Dense(1)
])
gru_model_reg.compile(optimizer=Adam(0.001), loss='mae')
gru_model_reg.fit(X_train_seq_scaled, y_train_reg_seq, epochs=3, batch_size=256, verbose=0)

In [490]:
gru_model_cla = Sequential([
    Input(shape=(WINDOW, len(features))),
    GRU(32),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])
gru_model_cla.compile(optimizer=Adam(0.001), loss='binary_crossentropy', metrics=['accuracy'])
gru_model_cla.fit(X_train_seq_scaled, y_train_cla_seq, epochs=3, batch_size=256, verbose=0)

In [491]:
gru_pred_reg = gru_model_reg.predict(X_test_seq_scaled, batch_size=512, verbose=0).flatten()
gru_pred_cla = (gru_model_cla.predict(X_test_seq_scaled, batch_size=512, verbose=0) > 0.5).astype(int).flatten()

In [492]:
print(f"Gru Regressor MAE (Next return prediction): {mean_absolute_error(y_test_reg_seq, gru_pred_reg):.6f}")
print(f"Gru Boost Classifier Accuracy (Classification prediction): {accuracy_score(y_test_cla_seq, gru_pred_cla):.4f}")

Gru Regressor MAE (Next return prediction): 0.024710
Gru Boost Classifier Accuracy (Classification prediction): 0.3284
